# Advanced PySpark Optimization — Examples 1–10

This notebook is the first of five notebooks containing 10 progressively more advanced examples each.

## Scope covered in this notebook

1. SparkSession and lazy evaluation
2. Jobs, stages, tasks, and DAG boundaries
3. Narrow vs. wide transformations
4. Reading CSV data and inspecting schemas/plans
5. RDD vs. DataFrame vs. SQL
6. Basic Spark SQL/DataFrame optimization
7. Execution-plan anatomy
8. `repartition()` vs. `coalesce()`
9. Measuring partition distribution
10. Choosing `spark.sql.shuffle.partitions`

### Source mapping

This notebook follows the terminology and organization from the source material, especially:
- SparkSession lifecycle and lazy execution
- jobs, stages, tasks, and DAGs
- narrow/wide transformations and shuffle boundaries
- DataFrame/SQL plan inspection
- partitioning and parallelism
- shuffle partition tuning

The examples intentionally use tiny CSV files so the mechanics are easy to inspect. The optimization patterns become more meaningful with larger datasets.

## Setup

The notebook expects this structure:

```text
pyspark_examples_01_10/
├── Advanced_PySpark_Examples_01_10.ipynb
└── data/
    ├── customers.csv
    └── orders.csv
```

Run this notebook from its own directory, or update `BASE_PATH` below.

In [ ]:
from pathlib import Path
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, DoubleType
)

# Update this path if you move the notebook/data files.
BASE_PATH = Path.cwd()
DATA_PATH = BASE_PATH / "data"

# Fallback for common notebook launch locations.
if not DATA_PATH.exists():
    candidate = Path("/mnt/data/pyspark_examples_01_10/data")
    if candidate.exists():
        DATA_PATH = candidate

print("Using data path:", DATA_PATH)
assert DATA_PATH.exists(), "Could not find the data directory. Update BASE_PATH."

spark = (
    SparkSession.builder
    .appName("Advanced-PySpark-Examples-01-10")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

print("Spark version:", spark.version)
print("Default shuffle partitions:", spark.conf.get("spark.sql.shuffle.partitions"))

# Example 1 — SparkSession and Lazy Evaluation

**Source concepts:** SparkSession lifecycle, transformations vs. actions, lazy execution.

A transformation creates a new logical plan. Spark does not execute that plan until an action is called.

In [ ]:
numbers_df = spark.range(1, 11).withColumnRenamed("id", "value")

lazy_df = (
    numbers_df
    .filter(F.col("value") > 3)
    .withColumn("value_x10", F.col("value") * 10)
)

print("No action has been called yet.")
print("The following explain() shows the plan Spark has built:")
lazy_df.explain()

print("\nNow an action triggers execution:")
lazy_df.show()

# Example 2 — Understanding Jobs, Stages, Tasks, and DAG Boundaries

**Source concepts:** jobs, stages, tasks, DAG understanding, stage boundaries.

Use `explain()` to inspect the physical plan. An `Exchange` generally indicates a shuffle boundary and therefore a stage boundary.

In [ ]:
# A simple aggregation creates a wide dependency and usually introduces a shuffle.
job_stage_df = (
    lazy_df
    .withColumn("bucket", F.col("value") % 2)
    .groupBy("bucket")
    .agg(F.sum("value_x10").alias("total"))
)

print("Formatted physical plan:")
job_stage_df.explain("formatted")

print("\nAction triggers execution of the plan:")
job_stage_df.show()

# Example 3 — Narrow vs. Wide Transformations

**Source concepts:** narrow transformations, wide transformations, shuffle operations.

A `filter()`/`select()` style operation is typically narrow. A `groupBy()` typically requires data redistribution and is wide.

In [ ]:
narrow_df = (
    numbers_df
    .filter(F.col("value") >= 5)
    .select("value")
)

wide_df = (
    numbers_df
    .withColumn("bucket", F.col("value") % 3)
    .groupBy("bucket")
    .count()
)

print("NARROW PLAN")
narrow_df.explain("formatted")

print("\nWIDE PLAN")
wide_df.explain("formatted")

print("\nExecuting both:")
narrow_df.show()
wide_df.show()

# Example 4 — Reading CSV Data, Schema Inspection, and Plans

**Source concepts:** DataFrame reads, schema handling, `explain()`, logical/physical planning.

The CSV files are deliberately small and readable.

In [ ]:
customers_schema = StructType([
    StructField("customer_id", StringType(), False),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("segment", StringType(), True),
])

orders_schema = StructType([
    StructField("order_id", StringType(), False),
    StructField("customer_id", StringType(), True),
    StructField("product", StringType(), True),
    StructField("category", StringType(), True),
    StructField("amount", DoubleType(), True),
    StructField("order_date", StringType(), True),
])

customers_df = (
    spark.read
    .option("header", True)
    .schema(customers_schema)
    .csv(str(DATA_PATH / "customers.csv"))
)

orders_df = (
    spark.read
    .option("header", True)
    .schema(orders_schema)
    .csv(str(DATA_PATH / "orders.csv"))
)

customers_df.printSchema()
orders_df.printSchema()

orders_df.show()

print("\nRead/filter plan:")
orders_df.filter(F.col("amount") > 500).explain("formatted")

# Example 5 — RDD vs. DataFrame vs. SQL

**Source concepts:** choosing the Spark abstraction; RDDs, DataFrames, and SQL.

The same logical result is expressed through three APIs.

In [ ]:
# RDD API
rdd_result = (
    orders_df.rdd
    .map(lambda row: (row.category, row.amount))
    .reduceByKey(lambda a, b: a + b)
)

print("RDD result:")
print(sorted(rdd_result.collect()))

# DataFrame API
df_result = (
    orders_df
    .groupBy("category")
    .agg(F.sum("amount").alias("total_amount"))
)

print("\nDataFrame result:")
df_result.show()

# SQL API
orders_df.createOrReplaceTempView("orders")

sql_result = spark.sql("""
    SELECT
        category,
        SUM(amount) AS total_amount
    FROM orders
    GROUP BY category
""")

print("\nSQL result:")
sql_result.show()

print("\nDataFrame physical plan:")
df_result.explain("formatted")

print("\nSQL physical plan:")
sql_result.explain("formatted")

# Example 6 — Basic DataFrame / SQL Optimization

**Source concepts:** filters, projections, built-in expressions, avoiding unnecessary data.

The example compares a less selective plan with a plan that filters and projects only the required columns.

In [ ]:
# Baseline: carry all columns until the aggregation.
baseline_df = (
    orders_df
    .groupBy("category")
    .agg(F.sum("amount").alias("total_amount"))
)

# Optimized expression: filter early and project only needed columns.
optimized_df = (
    orders_df
    .filter(F.col("amount") >= 500)
    .select("category", "amount")
    .groupBy("category")
    .agg(F.sum("amount").alias("total_amount"))
)

print("BASELINE PLAN")
baseline_df.explain("formatted")

print("\nOPTIMIZED PLAN")
optimized_df.explain("formatted")

print("\nOptimized result:")
optimized_df.show()

# Example 7 — Execution Plan Anatomy

**Source concepts:** logical plan, optimized logical plan, physical plan, Catalyst-oriented inspection.

`explain("extended")` shows the stages of plan construction more explicitly.

In [ ]:
plan_df = (
    orders_df
    .filter(F.col("category") == "Electronics")
    .withColumn("discounted_amount", F.col("amount") * F.lit(0.9))
    .select("customer_id", "product", "discounted_amount")
)

plan_df.explain("extended")

print("\nResult:")
plan_df.show()

# Example 8 — `repartition()` vs. `coalesce()`

**Source concepts:** increasing/reducing partitions, shuffle implications, partition count.

- `repartition()` can increase or decrease partitions and redistributes data.
- `coalesce()` is commonly used to reduce partitions with less data movement.

Inspect the partition counts and physical plans.

In [ ]:
original = orders_df

repartitioned = original.repartition(4, "customer_id")
coalesced = repartitioned.coalesce(2)

print("Original partitions:", original.rdd.getNumPartitions())
print("After repartition(4, customer_id):", repartitioned.rdd.getNumPartitions())
print("After coalesce(2):", coalesced.rdd.getNumPartitions())

print("\nREPARTITION PLAN")
repartitioned.explain("formatted")

print("\nCOALESCE PLAN")
coalesced.explain("formatted")

print("\nResults remain logically identical:")
print("Original:", original.count())
print("Repartitioned:", repartitioned.count())
print("Coalesced:", coalesced.count())

# Example 9 — Measuring Partition Distribution

**Source concepts:** partition distribution, imbalance, parallelism, skew detection.

For a small dataset, we intentionally create an uneven distribution by repartitioning on a low-cardinality/skewed key.

In [ ]:
skew_source = (
    orders_df
    .withColumn(
        "skew_key",
        F.when(F.col("customer_id").isin("C001", "C002", "C003"), F.lit("HOT"))
         .otherwise(F.lit("COLD"))
    )
)

partitioned_skew_df = skew_source.repartition(4, "skew_key")

def partition_counts(df):
    return (
        df.rdd
        .mapPartitionsWithIndex(
            lambda partition_id, rows: [(partition_id, sum(1 for _ in rows))]
        )
        .collect()
    )

print("Partition counts after repartition by skew_key:")
print(sorted(partition_counts(partitioned_skew_df)))

print("\nSkew key distribution:")
skew_source.groupBy("skew_key").count().show()

print("\nPhysical plan:")
partitioned_skew_df.explain("formatted")

# Example 10 — Choosing `spark.sql.shuffle.partitions`

**Source concepts:** shuffle partition count, task overhead, partition sizing, aggregation performance.

With only 10 records, this is a mechanics demonstration rather than a meaningful benchmark.
On real workloads, too few partitions can create oversized tasks; too many can create excessive scheduling overhead.

In [ ]:
def run_shuffle_partition_demo(num_partitions):
    spark.conf.set("spark.sql.shuffle.partitions", str(num_partitions))

    result = (
        orders_df
        .groupBy("category")
        .agg(
            F.sum("amount").alias("total_amount"),
            F.count("*").alias("order_count")
        )
    )

    print(f"\n--- spark.sql.shuffle.partitions = {num_partitions} ---")
    print("Configured value:", spark.conf.get("spark.sql.shuffle.partitions"))
    result.explain("formatted")
    result.show()

for partitions in [1, 2, 4, 8]:
    run_shuffle_partition_demo(partitions)

# Notebook 1 Summary

After these 10 examples, you should be comfortable with:

- SparkSession setup and lazy evaluation
- Jobs, stages, tasks, and shuffle boundaries
- Narrow vs. wide transformations
- Reading CSV data and applying explicit schemas
- RDD, DataFrame, and SQL APIs
- Early filtering and projection
- Reading logical and physical execution plans
- `repartition()` vs. `coalesce()`
- Measuring partition distribution
- Understanding `spark.sql.shuffle.partitions`

## Next notebook preview

Examples 11–20 will move into:

- partitioned writes with `partitionBy`
- partition pruning
- bucketing
- small-file considerations
- where shuffles come from
- aggregation behavior
- `groupByKey` vs. `reduceByKey`
- `rollup` and `cube`
- reducing shuffle through better data layout